In [1]:
import os

os.chdir("..")

In [2]:
import json
from pathlib import Path
from examples.legal_hybrid_rag import build_pipeline, validate_ingest_coverage

root = Path(".").resolve()
print(root)
docs_dir = root / "docs_corpus"
ingest_root = root / "ingestion" / "docs_corpus_ingest_result"
validate_ingest_coverage(docs_dir, ingest_root)
questions = json.loads((root / "questions.json").read_text())

/home/duongkstn/miniconda3/envs/py312/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/home/duongkstn/Downloads/agentic-rag-legal-challenge


In [5]:
from arlc import get_config
CONFIG = get_config()

In [3]:
from retrieval import LegalHybridRAGPipeline
from pathlib import Path
ROOT_DIR = root

def build_pipeline2():
    from langchain_openai import ChatOpenAI, OpenAIEmbeddings
    from langchain_google_genai import GoogleGenerativeAI, GoogleGenerativeAIEmbeddings

    llm = GoogleGenerativeAI(
        model="gemini-2.5-flash-lite",
        temperature=0.0,
        google_api_key=CONFIG.get_llm_api_key(),
        # google_api_base=CONFIG.llm_api_base,
    )
    embeddings = GoogleGenerativeAIEmbeddings(
        model="gemini-embedding-2-preview",
        google_api_key=CONFIG.get_embedding_api_key(),
        # google_api_base=CONFIG.llm_api_base,
    )

    return LegalHybridRAGPipeline(
        llm=llm,
        embedding_model=embeddings,
        ingest_root=str(ROOT_DIR / "ingestion" / "docs_corpus_ingest_result"),
        docs_root=str(Path(CONFIG.docs_dir)),
        chroma_persist_dir=str(ROOT_DIR / "tmp" / "legal_hybrid_chroma"),
        use_persistent_db=True,
    )

In [6]:
pipeline = build_pipeline2()

In [7]:
pipeline.build_indexes()
pipeline.build_indexes_pyserini()

/home/duongkstn/Downloads/agentic-rag-legal-challenge/retrieval/legal_hybrid_rag_pipeline.py:137: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  self.doc_search = Chroma(
Mar 16, 2026 1:01:33 AM org.apache.lucene.store.MemorySegmentIndexInputProvider <init>
INFO: Using MemorySegmentIndexInput with Java 21; to disable start with -Dorg.apache.lucene.store.MMapDirectory.enableMemorySegments=false


In [7]:
len(pipeline.source_documents)

30

In [8]:
len(pipeline.raw_chunks)

3132

In [9]:
j = 21
chunk = pipeline.raw_chunks[j]
print(chunk.metadata['doc_id'])
print(chunk.page_content)

03b621728fe29eb6113fcdb57f6458d793fd2d5c5b833ae26d40f04a29c85359
CA 005/2025 | LXT Real Estate Broker L.L.C v SIR Real Estate LLC | BETWEEN > 5. The CFI is to deal with the remitted application for security on the basis that: | chunk=section

5. The CFI is to deal with the remitted application for security on the basis that:
(a) the jurisdictional conditions for the grant of security have been satisfi ed; and (b) the issues identifi ed in Order 3and the issues pertaining to the proposed cross appeal have been determined adversely to the Claimant as at the time of the original application for security
For the avoidance of doubt, nothing in these orders prevents the Claimant from raising any argument based on the Crabtree principle if and when the question of security is reconsidered following the fi ling of a counterclaim by the Defendant.


In [7]:
pipeline.pyserini_retriever

In [12]:
from openai import OpenAI, AsyncOpenAI
import asyncio
from ragas.llms import llm_factory
from ragas.metrics.collections import ContextRelevance
client = AsyncOpenAI(
    api_key=CONFIG.get_llm_api_key(),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)
llm = llm_factory(model="gemini-2.5-flash-lite", client=client)
scorer = ContextRelevance(llm=llm)

async def get_score(question, contexts):
    result = await scorer.ascore(
        user_input=question,
        retrieved_contexts=contexts
    )
    return result


In [19]:
asyncio.run(get_score("what is capital of Vietnam", ["Hanoi is capital of Vietnam"]))

MetricResult(value=1.0)

In [15]:
import nest_asyncio
nest_asyncio.apply()
from tqdm import tqdm

def evaluate(questions):
    scores = []
    for question_item in tqdm(questions):
        # result = pipeline.answer_question(question)
        question_text = question_item["question"]
        answer_type = question_item.get("answer_type", "free_text")
        route = pipeline.router.route(question_text, answer_type)

        if route.comparison_mode and len(route.case_ids) > 1:
            supporting_docs = pipeline._retrieve_for_comparison(question_text, answer_type, route)
        else:
            supporting_docs, route = pipeline.retrieve(question_text, answer_type=answer_type, route=route)

        contexts = [chunk.page_content for chunk in supporting_docs]
        # print("\nQUESTION:", question_item["id"])
        # print(question_text)
        # print("REFS:", [chunk.__dict__ for chunk in supporting_docs])
        score = asyncio.run(get_score(question_text, contexts))
        scores.append(score)
    return scores
scores = evaluate(questions)
print(scores)
avg_score = sum(scores) / len(scores)
print(avg_score)

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [16:53<00:00, 10.13s/it]

[MetricResult(value=1.0), MetricResult(value=1.0), MetricResult(value=1.0), MetricResult(value=0.5), MetricResult(value=1.0), MetricResult(value=0.5), MetricResult(value=1.0), MetricResult(value=0.25), MetricResult(value=1.0), MetricResult(value=0.5), MetricResult(value=1.0), MetricResult(value=0.75), MetricResult(value=0.5), MetricResult(value=0.5), MetricResult(value=0.5), MetricResult(value=1.0), MetricResult(value=0.5), MetricResult(value=1.0), MetricResult(value=1.0), MetricResult(value=1.0), MetricResult(value=0.5), MetricResult(value=1.0), MetricResult(value=1.0), MetricResult(value=0.75), MetricResult(value=1.0), MetricResult(value=1.0), MetricResult(value=1.0), MetricResult(value=1.0), MetricResult(value=0.5), MetricResult(value=1.0), MetricResult(value=1.0), MetricResult(value=0.5), MetricResult(value=1.0), MetricResult(value=0.5), MetricResult(value=1.0), MetricResult(value=1.0), MetricResult(value=1.0), MetricResult(value=1.0), MetricResult(value=0.5), MetricResult(value=0.

In [8]:

for item in questions[:3]:
    result = pipeline.answer_question(item)
    print("\nQUESTION:", item["id"])
    print(item["question"])
    print("ANSWER:", result.answer)
    print("REFS:", [ref.__dict__ for ref in result.retrieval_refs])



RRRRRRRRRRRR RetrievalResult(documents=[Document(metadata={'doc_id': '03b621728fe29eb6113fcdb57f6458d793fd2d5c5b833ae26d40f04a29c85359', 'source': '/home/duongkstn/Downloads/agentic-rag-legal-challenge/docs_corpus/03b621728fe29eb6113fcdb57f6458d793fd2d5c5b833ae26d40f04a29c85359.pdf', 'source_path': '/home/duongkstn/Downloads/agentic-rag-legal-challenge/docs_corpus/03b621728fe29eb6113fcdb57f6458d793fd2d5c5b833ae26d40f04a29c85359.pdf', 'claim_number': 'CA 005/2025', 'case_name': 'LXT Real Estate Broker L.L.C v SIR Real Estate LLC', 'court': 'THE DUBAI INTERNATIONAL FINANCIAL CENTRE COURTS', 'court_division': 'COURT OF APPEAL', 'judgment_date': 'JANUARY 13, 2026', 'judgment_release_date': '', 'hearing_date': '', 'claimant': 'LXT REAL ESTATE BROKER L.L.C', 'defendants': ['SIR REAL ESTATE LLC'], 'neutral_citation': '', 'claimant_counsel': [], 'defendant_counsel': [], 'claimant_law_firm': '', 'defendant_law_firm': '', 'doc_type': 'case_judgment', 'structure_available': True, 'block_count': 3